# CoxPH Algorithm Test using the MockNetwork
The mock network is intended for testing algorithms without the need of setting up an 
entire vantage6 network.

In [ ]:
import pandas as pd

## MockClient
We mock to have two organizations with three databases each (<code>rps_cohort</code>, <code>pelvis_cohort</code> and <code>rps_pelvis_cohort</code>). <br><br>
The first organization has:
* <code>rps_cohort[:10]</code> (10 pts)
* <code>pelvis_cohort[:10]</code> (10 pts)
* <code>rps_pelvis_cohort[:10]</code> (10 pts)

The second organization has:
* <code>rps_cohort[10:]</code> (304 pts)
* <code>pelvis_cohort[10:]</code> (276 pts)
* <code>non_liposarcoma_cohort[10:]</code> (590 pts)

In [ ]:
# Load the dataframes from parquet files
rps_cohort = pd.read_parquet("../data/rps_cohort.parquet")
pelvis_cohort = pd.read_parquet("../data/pelvis_cohort.parquet")
rps_pelvis_cohort = pd.read_parquet("../data/rps_pelvis_cohort.parquet")


In [ ]:
# Temporary set the survival_days to the delta between the surgery date and today
rps_cohort["survival_days"] = (pd.Timestamp("2026-01-01", tz="UTC") - rps_cohort["surgery_date"]).dt.days.astype("Int64")
pelvis_cohort["survival_days"] = (pd.Timestamp("2026-01-01", tz="UTC") - pelvis_cohort["surgery_date"]).dt.days.astype("Int64")
rps_pelvis_cohort["survival_days"] = (pd.Timestamp("2026-01-01", tz="UTC") - rps_pelvis_cohort["surgery_date"]).dt.days.astype("Int64")


In [ ]:
from vantage6.algorithm.mock.network import MockNetwork

network = MockNetwork(
    "v6-analytics",
    datasets=[
        {
            "rps": {"database": rps_cohort[:20], "db_type": "omop"},
            "pelvis": {"database": pelvis_cohort[:20], "db_type": "omop"},
            "rps_pelvis": {"database": rps_pelvis_cohort[:20], "db_type": "omop"}
        },
        {
            "rps": {"database": rps_cohort[20:], "db_type": "omop"},
            "pelvis": {"database": pelvis_cohort[20:], "db_type": "omop"},
            "rps_pelvis": {"database": rps_pelvis_cohort[20:], "db_type": "omop"}
        }
    ],
    collaboration_id=1,
)

client = network.user_client

In [ ]:
network.get_node(2).dataframes['rps'].head()

In [ ]:
# overall summary results
task = client.task.create(
    method="coxph_central",
    organizations=[1],
    arguments={
        "time_col": "survival_days",
        "outcome_col": "tumor_rupture",	
        "expl_vars": ["age", "sex"],
        "organizations_to_include": [1,2]
    },
    databases=[
        [
            {"type": "dataframe", "dataframe_id": 1},
            {"type": "dataframe", "dataframe_id": 2},
            {"type": "dataframe", "dataframe_id": 3}
        ]
    ],
    action="central_compute"
)

In [ ]:
results = client.result.from_task(task_id=task.get("id"))
results

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

cohorts = results[0]["cohorts"]
rows = []
for cohort_name, data in cohorts.items():
    model = json.loads(data["model"])
    for var in model["Coef"]:
        rows.append({
            "cohort": cohort_name,
            "variable": var,
            "HR": model["Exp(coef)"][var],
            "lower_CI": model["lower_CI"][var],
            "upper_CI": model["upper_CI"][var],
            "p_value": model["p-value"][var],
        })

df = pd.DataFrame(rows)
df

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

y_pos = np.arange(len(df))
ax.errorbar(
    df["HR"],
    y_pos,
    xerr=[df["HR"] - df["lower_CI"], df["upper_CI"] - df["HR"]],
    fmt="o",
    capsize=4,
    capthick=1.5,
)

ax.axvline(1, color="gray", linestyle="--", linewidth=1)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"{r.cohort} ({r.variable})" for _, r in df.iterrows()])
ax.set_xlabel("Hazard ratio (95% CI)")
ax.set_title("Cox PH: age effect by cohort")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
display_df = df.assign(
    HR_95CI=df.apply(lambda r: f"{r['HR']:.3f} ({r['lower_CI']:.3f}–{r['upper_CI']:.3f})", axis=1),
    p_value=df["p_value"].round(4),
)[["cohort", "variable", "HR_95CI", "p_value"]]
display_df

In [ ]:
task = client.task.create(
    method="coxph_central",
    organizations=[1],
    arguments={
        "time_col": "survival_days",
        "outcome_col": "tumor_rupture",	
        "expl_vars": ["age"],
        "organizations_to_include": [1,2]
    },
    databases=[
        [
            {"type": "dataframe", "dataframe_id": 1},
            {"type": "dataframe", "dataframe_id": 2},
            {"type": "dataframe", "dataframe_id": 3}
        ]
    ],
    action="central_compute"
)

In [ ]:
task = client.task.create(
    method="coxph_central",
    organizations=[1],
    arguments={
        "time_col": "survival_days",
        "outcome_col": "tumor_rupture",	
        "expl_vars": ["sex"],
        "organizations_to_include": [1,2]
    },
    databases=[
        [
            {"type": "dataframe", "dataframe_id": 1},
            {"type": "dataframe", "dataframe_id": 2},
            {"type": "dataframe", "dataframe_id": 3}
        ]
    ],
    action="central_compute"
)